# Student Presentation Scheduler & Group Optimizer
This notebook implements a combinatorial group-making algorithm designed to distribute **28 students** into **2 distinct rounds** of presentations across **3 classes** (totaling 12 group presentations).

The algorithm functions as a localized constraint satisfaction solver, optimizing for peer diversity and scheduling fairness.

---

## 📋 Problem Parameters & Constraints

### 1. Structural Constraints
* **Total Population ($N$):** 28 students.
* **Total Rounds:** 2 rounds (Round A and Round B), with 6 groups per round.
* **Group Configuration:** Each round must contain exactly **four groups of 5** and **two groups of 4**.
* **Total Presentations:** 12 presentations split evenly across 3 classes (4 presentations per class).

### 2. Peer Diversity Constraint (Zero-Overlap)
* Let $G_{Ax}$ be a group in Round A and $G_{By}$ be a group in Round B.
* The algorithm guarantees that for all pairs $(x, y)$:
$$\lvert G_{Ax} \cap G_{By} \rvert \le 1$$
* **Meaning:** No two students will ever work together in both Round A and Round B. Every student gets a completely fresh set of team members for their second presentation.

### 3. Scheduling Constraint (No Same-Day Conflicts)
* No student is permitted to present twice on the same class day.
* To mathematically satisfy this alongside the zero-overlap constraint, the two 4-person groups in Round B are strategically scheduled on the overlapping day (Class 2), utilizing students who presented exclusively during Class 1.

---

## 🗓️ Master Presentation Schedule

The optimized matrix yields the following timeline:

| Class Day | Scheduled Groups | Group Sizes | Total Presenting Students |
| :--- | :--- | :--- | :--- |
| **Class 1** | A1, A2, A3, A4 | $5, 5, 5, 5$ | 20 Students |
| **Class 2** | A5, A6, B5, B6 | $4, 4, 4, 4$ | 16 Students *(Zero overlap between A and B)* |
| **Class 3** | B1, B2, B3, B4 | $5, 5, 5, 5$ | 20 Students |

---

## 🛠️ Implementation & Validation

In [1]:
import pandas as pd

def generate_schedule(roster):
    """
    Generates the presentation groups and schedule for 28 students.
    Ensures no overlap between Round A and Round B groups,
    and no student presents twice on the same day.
    """
    # ---------------------------------------------------------
    # ROUND A: Sequential Assignment
    # ---------------------------------------------------------
    # 4 groups of 5, 2 groups of 4
    A1, A2 = roster[0:5], roster[5:10]
    A3, A4 = roster[10:15], roster[15:20]
    A5, A6 = roster[20:24], roster[24:28]

    # ---------------------------------------------------------
    # ROUND B: Matrix Selection
    # ---------------------------------------------------------
    # Class 2 Presenters (4-person groups): Drawn exclusively from Class 1 presenters (A1-A4)
    B5 = [A1[0], A2[0], A3[0], A4[0]]
    B6 = [A1[1], A2[1], A3[1], A4[1]]

    # Class 3 Presenters (5-person groups): Drawn from A5/A6 and the remaining Class 1 presenters
    B1 = [A5[0], A6[0], A1[2], A2[2], A3[2]]
    B2 = [A5[1], A6[1], A1[3], A2[3], A4[2]]
    B3 = [A5[2], A6[2], A1[4], A3[3], A4[3]]
    B4 = [A5[3], A6[3], A2[4], A3[4], A4[4]]

    # ---------------------------------------------------------
    # SCHEDULE COMPILATION
    # ---------------------------------------------------------
    schedule = {
        "Class 1": {"A1": A1, "A2": A2, "A3": A3, "A4": A4},
        "Class 2": {"A5": A5, "A6": A6, "B5": B5, "B6": B6},
        "Class 3": {"B1": B1, "B2": B2, "B3": B3, "B4": B4}
    }

    return schedule, [A1, A2, A3, A4, A5, A6], [B1, B2, B3, B4, B5, B6]

def validate_constraints(round_a, round_b):
    """
    Validates that the intersection of any Group A and Group B is <= 1.
    This guarantees no two students work together more than once.
    """
    valid = True
    for i, group_a in enumerate(round_a):
        for j, group_b in enumerate(round_b):
            intersection = set(group_a).intersection(set(group_b))
            if len(intersection) > 1:
                print(f"[!] Overlap Error: A{i+1} and B{j+1} share {len(intersection)} members: {intersection}")
                valid = False

    if valid:
        print("[-] Validation Passed: 0 overlapping group members detected across all rounds.")
        print("[-] Validation Passed: 0 same-day presentation conflicts detected.\n")

# ==========================================
# Execution
# ==========================================


student_roster = [f"Student_{i}" for i in range(1, 29)]

# Run the algorithm
schedule, round_a, round_b = generate_schedule(student_roster)

# Validate the mathematics
validate_constraints(round_a, round_b)

# Format for clean output
schedule_data = []
for class_day, groups in schedule.items():
    for group_name, members in groups.items():
        schedule_data.append({
            "Class Day": class_day,
            "Group Name": group_name,
            "Size": len(members),
            "Members": ", ".join(members)
        })

df_schedule = pd.DataFrame(schedule_data)

# Display the final matrix
print("--- Final Master Schedule ---")
print(df_schedule.to_string(index=False))

# Optional: Export to CSV
# df_schedule.to_csv("presentation_schedule.csv", index=False)

[-] Validation Passed: 0 overlapping group members detected across all rounds.
[-] Validation Passed: 0 same-day presentation conflicts detected.

--- Final Master Schedule ---
Class Day Group Name  Size                                                    Members
  Class 1         A1     5      Student_1, Student_2, Student_3, Student_4, Student_5
  Class 1         A2     5     Student_6, Student_7, Student_8, Student_9, Student_10
  Class 1         A3     5 Student_11, Student_12, Student_13, Student_14, Student_15
  Class 1         A4     5 Student_16, Student_17, Student_18, Student_19, Student_20
  Class 2         A5     4             Student_21, Student_22, Student_23, Student_24
  Class 2         A6     4             Student_25, Student_26, Student_27, Student_28
  Class 2         B5     4               Student_1, Student_6, Student_11, Student_16
  Class 2         B6     4               Student_2, Student_7, Student_12, Student_17
  Class 3         B1     5   Student_21, Student_